# Power Apps YAML Validator

Validate `.pa.yaml` canvas source in Jupyter with **Studio-style diagnostics** (PA2108 unknown property, PA2109 invalid variant, and related import traps).

**Studio parity scope:** catalogued controls get version-aware property allowlists and trap-table checks. Unknown control types (for example custom components not in the catalog) skip the property matrix so we do not invent false PA2108 errors.

**Example:** `RadiusTopLeft` on `Label@2.5.1` triggers PA2108; the safe repair is to **remove the property line** (Label has no radius—use a `GroupContainer` wrapper if you need rounded corners).

References: Microsoft Learn Power Apps YAML source code, `nfBi(EN,FR)` bilingual formulas, `|-` multiline Power Fx, responsive `App`/`Parent` sizing, and accessible labels/tooltips.


In [6]:
# Section 1: Set Up Notebook Environment and Dependencies
%pip install -q -r ../requirements.txt

import ipywidgets as widgets
import yaml
import jsonschema
from IPython.display import display


from pathlib import Path
import sys

# Section 2: Load the Provided Source and Create a Baseline Run
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from powerapps_yaml_validator import (
    FixApplication,
    PowerAppsYamlValidatorUI,
    apply_fixes,
    create_validator_ui,
    propose_fixes,
    validate_text,
)

sample_yaml = """Screens:
  All Training Sessions | Toutes les séances de formation:
    Properties:
      Fill: =myTheme.Background
      OnVisible: |-
        =UpdateContext(
            {
                locSessionsShowCommunities: false,
                locSessionsCommunityFilters: Blank(),
                locSessionsSelectedSession: Blank(),
                locSessionsSortAscending: true,
                locSessionsRefreshedAt: Now()
            }
        )
    Children:
      - modern_btn_Validate:
          Control: ModernButton
          Properties:
            Text: =nfBi("Validate source", "Valider la source")
      - sessions_con_Root:
          Control: GroupContainer@1.5.0
          Variant: AutoLayout
          Properties:
            DropShadow: =DropShadow.None
            Fill: =myTheme.Background
            Height: =App.Height
            LayoutAlignItems: =LayoutAlignItems.Stretch
            LayoutDirection: =LayoutDirection.Horizontal
            LayoutMinHeight: =0
            LayoutMinWidth: =0
            Width: =App.Width
          Children:
            - sessions_cmp_SideMenu:
                Control: CanvasComponent
                ComponentName: _cx_nav_SideMenu
                Properties:
                  AppName: =nfBi("HR Learning Hub", "Carrefour d'apprentissage des RH")
                  CollapsedWidth: =75
                  ExpandedWidth: =320
                  Fill: =myTheme.Transparent
                  Height: =App.Height
                  HelpURL: =""
                  Logo: =img.Home
                  LogoCompact: =img.Report
                  MenuItems: =tbl.Menus
                  ProfileImage: =User().Image
                  ProfileName: =User().FullName
                  ProfileRole: =nfBi("Employee", "Employé")
                  Width: |-
                    =If(
                        SideMenuCollapsed,
                        Self.CollapsedWidth,
                        Self.ExpandedWidth
                    )
                  svgMode: =true
            - sessions_con_Main:
                Control: GroupContainer@1.5.0
                Variant: AutoLayout
                Properties:
                  DropShadow: =DropShadow.None
                  Fill: =myTheme.Background
                  FillPortions: =1
                  LayoutAlignItems: =LayoutAlignItems.Stretch
                  LayoutDirection: =LayoutDirection.Vertical
                  LayoutGap: =10
                  LayoutMinHeight: =0
                  LayoutMinWidth: =0
                  PaddingBottom: =10
                  PaddingLeft: =16
                  PaddingRight: =16
                  PaddingTop: =8
                Children:
                  - sessions_cmp_HeaderBar:
                      Control: CanvasComponent
                      ComponentName: _cx_nav_HeaderBar
                      Properties:
                        AlignInContainer: =AlignInContainer.Stretch
                        Fill: =myTheme.Transparent
                        Height: =60
                        LayoutMinHeight: =60
                        LayoutMinWidth: =0
                        NavItems: =tbl.Tabs.Requests
                        Width: =Parent.Width
                        svgMode: =true
                  - sessions_con_Hero:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Surface
                        FillPortions: =0
                        Height: =108
                        LayoutAlignItems: =LayoutAlignItems.Center
                        LayoutDirection: =LayoutDirection.Horizontal
                        LayoutGap: =16
                        LayoutMinHeight: =108
                        LayoutMinWidth: =0
                        PaddingBottom: =16
                        PaddingLeft: =20
                        PaddingRight: =20
                        PaddingTop: =16
                        RadiusBottomLeft: =12
                        RadiusBottomRight: =12
                        RadiusTopLeft: =12
                        RadiusTopRight: =12
                      Children:
                        - sessions_con_HeroText:
                            Control: GroupContainer@1.5.0
                            Variant: AutoLayout
                            Properties:
                              DropShadow: =DropShadow.None
                              Fill: =myTheme.Transparent
                              FillPortions: =1
                              LayoutDirection: =LayoutDirection.Vertical
                              LayoutGap: =4
                              LayoutMinHeight: =0
                              LayoutMinWidth: =240
                            Children:
                              - sessions_lbl_Title:
                                  Control: Label@2.5.1
                                  Properties:
                                    AlignInContainer: =AlignInContainer.Stretch
                                    Color: =myTheme.Text
                                    Fill: =myTheme.Transparent
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Bold
                                    Height: =40
                                    LayoutMinHeight: =40
                                    LayoutMinWidth: =0
                                    PaddingBottom: =0
                                    PaddingLeft: =0
                                    PaddingRight: =0
                                    PaddingTop: =0
                                    Size: =22
                                    Text: =nfBi("Upcoming training sessions", "Séances de formation à venir")
                              - sessions_lbl_Subtitle:
                                  Control: Label@2.5.1
                                  Properties:
                                    AlignInContainer: =AlignInContainer.Stretch
                                    Color: =myTheme.TextMuted
                                    Fill: =myTheme.Transparent
                                    Font: =Font.Lato
                                    Height: =34
                                    LayoutMinHeight: =34
                                    LayoutMinWidth: =0
                                    PaddingBottom: =0
                                    PaddingLeft: =0
                                    PaddingRight: =0
                                    PaddingTop: =0
                                    Size: =11
                                    Text: |-
                                      =nfBi(
                                          "Find scheduled learning by community, region, district, language and course category.",
                                          "Trouvez les formations prévues par communauté, région, district, langue et catégorie de cours."
                                      )
                        - sessions_con_Summary:
                            Control: GroupContainer@1.5.0
                            Variant: AutoLayout
                            Properties:
                              DropShadow: =DropShadow.None
                              Fill: =myTheme.InformationBackground
                              FillPortions: =0
                              Height: =64
                              LayoutAlignItems: =LayoutAlignItems.Center
                              LayoutDirection: =LayoutDirection.Horizontal
                              LayoutGap: =10
                              LayoutMinHeight: =64
                              LayoutMinWidth: =210
                              PaddingLeft: =14
                              PaddingRight: =14
                              RadiusBottomLeft: =10
                              RadiusBottomRight: =10
                              RadiusTopLeft: =10
                              RadiusTopRight: =10
                              Width: =230
                            Children:
                              - sessions_ico_Summary:
                                  Control: Classic/Icon@2.5.0
                                  Properties:
                                    AccessibleLabel: =nfBi("Scheduled sessions", "Séances prévues")
                                    Color: =myTheme.InformationBackgroundText
                                    Height: =28
                                    HoverColor: =myTheme.InformationBackgroundText
                                    Icon: =Icon.CalendarBlank
                                    LayoutMinHeight: =28
                                    LayoutMinWidth: =28
                                    PressedColor: =myTheme.InformationBackgroundText
                                    TabIndex: =0
                                    Tooltip: =nfBi("Scheduled sessions", "Séances prévues")
                                    Width: =28
                              - sessions_con_SummaryText:
                                  Control: GroupContainer@1.5.0
                                  Variant: AutoLayout
                                  Properties:
                                    DropShadow: =DropShadow.None
                                    Fill: =myTheme.Transparent
                                    FillPortions: =1
                                    LayoutDirection: =LayoutDirection.Vertical
                                    LayoutGap: =0
                                    LayoutMinHeight: =0
                                    LayoutMinWidth: =100
                                  Children:
                                    - sessions_lbl_SummaryValue:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.InformationBackgroundText
                                          Fill: =myTheme.Transparent
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =30
                                          LayoutMinHeight: =30
                                          LayoutMinWidth: =0
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =18
                                          Text: =Text(sessions_gal_Results.AllItemsCount, "#,##0")
                                    - sessions_lbl_SummaryLabel:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.InformationBackgroundText
                                          Fill: =myTheme.Transparent
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Semibold
                                          Height: =22
                                          LayoutMinHeight: =22
                                          LayoutMinWidth: =0
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =9
                                          Text: =nfBi("Matching sessions", "Séances correspondantes")
                  - sessions_con_Search:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Surface
                        FillPortions: =0
                        Height: =60
                        LayoutAlignItems: =LayoutAlignItems.Center
                        LayoutDirection: =LayoutDirection.Horizontal
                        LayoutGap: =10
                        LayoutMinHeight: =60
                        LayoutMinWidth: =0
                        PaddingBottom: =8
                        PaddingLeft: =12
                        PaddingRight: =12
                        PaddingTop: =8
                        RadiusBottomLeft: =10
                        RadiusBottomRight: =10
                        RadiusTopLeft: =10
                        RadiusTopRight: =10
                      Children:
                        - sessions_ico_Search:
                            Control: Classic/Icon@2.5.0
                            Properties:
                              AccessibleLabel: =nfBi("Search", "Rechercher")
                              Color: =myTheme.IconsMuted
                              Height: =24
                              HoverColor: =myTheme.Icons
                              Icon: =Icon.Search
                              LayoutMinHeight: =24
                              LayoutMinWidth: =24
                              PressedColor: =myTheme.Icons
                              TabIndex: =0
                              Tooltip: =nfBi("Search training sessions", "Rechercher des séances de formation")
                              Width: =24
                        - sessions_txt_Search:
                            Control: Classic/TextInput@2.3.2
                            Properties:
                              AccessibleLabel: =nfBi("Search training sessions", "Rechercher des séances de formation")
                              BorderColor: =myTheme.Border
                              BorderThickness: =1
                              Color: =myTheme.InputText
                              Default: =""
                              DisabledBorderColor: =myTheme.Border
                              DisabledColor: =myTheme.DisabledText
                              DisabledFill: =myTheme.InputDisabled
                              Fill: =myTheme.Input
                              FillPortions: =1
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =42
                              HintText: =nfBi("Search by course, session or location", "Rechercher par cours, séance ou emplacement")
                              HoverBorderColor: =myTheme.BorderStrong
                              HoverColor: =myTheme.InputText
                              HoverFill: =myTheme.InputHover
                              LayoutMinHeight: =42
                              LayoutMinWidth: =200
                              PaddingLeft: =12
                              RadiusBottomLeft: =8
                              RadiusBottomRight: =8
                              RadiusTopLeft: =8
                              RadiusTopRight: =8
                              Size: =11
                        - sessions_ico_Sort:
                            Control: Classic/Icon@2.5.0
                            Properties:
                              AccessibleLabel: |-
                                =If(
                                    locSessionsSortAscending,
                                    nfBi("Sort newest first", "Trier les plus récentes en premier"),
                                    nfBi("Sort earliest first", "Trier les plus anciennes en premier")
                                )
                              Color: =myTheme.Icons
                              Height: =36
                              HoverColor: =myTheme.SecondaryHover
                              Icon: =If(locSessionsSortAscending, Icon.SortUp, Icon.SortDown)
                              LayoutMinHeight: =36
                              LayoutMinWidth: =36
                              OnSelect: |-
                                =UpdateContext({locSessionsSortAscending: !locSessionsSortAscending})
                              PressedColor: =myTheme.SecondaryPressed
                              TabIndex: =0
                              Tooltip: |-
                                =If(
                                    locSessionsSortAscending,
                                    nfBi("Sort newest first", "Trier les plus récentes en premier"),
                                    nfBi("Sort earliest first", "Trier les plus anciennes en premier")
                                )
                              Width: =36
                        - sessions_ico_Reset:
                            Control: Classic/Icon@2.5.0
                            Properties:
                              AccessibleLabel: =nfBi("Reset all filters", "Réinitialiser tous les filtres")
                              Color: =myTheme.Icons
                              Height: =36
                              HoverColor: =myTheme.ClearHover
                              Icon: =Icon.Reload
                              LayoutMinHeight: =36
                              LayoutMinWidth: =36
                              OnSelect: |-
                                =Reset(sessions_txt_Search);
                                Reset(sessions_cmb_Region);
                                Reset(sessions_cmb_District);
                                Reset(sessions_cmb_Language);
                                Reset(sessions_cmb_Category);
                                UpdateContext(
                                    {
                                        locSessionsCommunityFilters: Blank(),
                                        locSessionsSelectedSession: Blank()
                                    }
                                )
                              PressedColor: =myTheme.ClearPressed
                              TabIndex: =0
                              Tooltip: =nfBi("Reset all filters", "Réinitialiser tous les filtres")
                              Width: =36
                  - sessions_con_Workspace:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Transparent
                        FillPortions: =1
                        LayoutAlignItems: =LayoutAlignItems.Stretch
                        LayoutDirection: =If(App.Width < 1050, LayoutDirection.Vertical, LayoutDirection.Horizontal)
                        LayoutGap: =12
                        LayoutMinHeight: =0
                        LayoutMinWidth: =0
                      Children:
                        - sessions_con_FilterPanel:
                            Control: GroupContainer@1.5.0
                            Variant: AutoLayout
                            Properties:
                              DropShadow: =DropShadow.None
                              Fill: =myTheme.Surface
                              FillPortions: =0
                              Height: =If(App.Width < 1050, 330, Parent.Height)
                              LayoutDirection: =LayoutDirection.Vertical
                              LayoutGap: =10
                              LayoutMinHeight: =300
                              LayoutMinWidth: =260
                              LayoutOverflowY: =LayoutOverflow.Scroll
                              PaddingBottom: =16
                              PaddingLeft: =16
                              PaddingRight: =16
                              PaddingTop: =16
                              RadiusBottomLeft: =12
                              RadiusBottomRight: =12
                              RadiusTopLeft: =12
                              RadiusTopRight: =12
                              Width: =If(App.Width < 1050, Parent.Width, Min(330, Parent.Width * 0.28))
                            Children:
                              - sessions_con_FilterHeader:
                                  Control: GroupContainer@1.5.0
                                  Variant: AutoLayout
                                  Properties:
                                    DropShadow: =DropShadow.None
                                    Fill: =myTheme.Transparent
                                    FillPortions: =0
                                    Height: =38
                                    LayoutAlignItems: =LayoutAlignItems.Center
                                    LayoutDirection: =LayoutDirection.Horizontal
                                    LayoutGap: =8
                                    LayoutMinHeight: =38
                                    LayoutMinWidth: =0
                                  Children:
                                    - sessions_ico_Filter:
                                        Control: Classic/Icon@2.5.0
                                        Properties:
                                          AccessibleLabel: =nfBi("Filters", "Filtres")
                                          Color: =myTheme.Icons
                                          Height: =24
                                          Icon: =Icon.Filter
                                          LayoutMinHeight: =24
                                          LayoutMinWidth: =24
                                          TabIndex: =0
                                          Tooltip: =nfBi("Session filters", "Filtres de séance")
                                          Width: =24
                                    - sessions_lbl_FilterHeading:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.Text
                                          Fill: =myTheme.Transparent
                                          FillPortions: =1
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =32
                                          LayoutMinHeight: =32
                                          LayoutMinWidth: =100
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =14
                                          Text: =nfBi("Refine results", "Affiner les résultats")
                              - sessions_btn_Communities:
                                  Control: Classic/Button@2.2.0
                                  Properties:
                                    Align: =Align.Left
                                    AlignInContainer: =AlignInContainer.Stretch
                                    BorderColor: =myTheme.Secondary
                                    BorderStyle: =BorderStyle.Solid
                                    BorderThickness: =1
                                    Color: =myTheme.Secondary
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.Transparent
                                    Fill: =myTheme.Transparent
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Semibold
                                    Height: =48
                                    HoverBorderColor: =myTheme.SecondaryHover
                                    HoverColor: =myTheme.SecondaryText
                                    HoverFill: =myTheme.SecondaryHover
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    OnSelect: |-
                                      =UpdateContext({locSessionsShowCommunities: true})
                                    PaddingLeft: =12
                                    PaddingRight: =12
                                    PressedBorderColor: =myTheme.SecondaryPressed
                                    PressedColor: =myTheme.SecondaryText
                                    PressedFill: =myTheme.SecondaryPressed
                                    RadiusBottomLeft: =8
                                    RadiusBottomRight: =8
                                    RadiusTopLeft: =8
                                    RadiusTopRight: =8
                                    Size: =10
                                    Text: |-
                                      =nfBi("Target communities: ", "Communautés cibles : ") &
                                      Coalesce(
                                          Concat(
                                              Filter(
                                                  Table(
                                                      {Value: nfBi("BSOs", "ASF"), Applied: locSessionsCommunityFilters.BSO},
                                                      {Value: nfBi("Hearings advisors", "Conseillers aux audiences"), Applied: locSessionsCommunityFilters.HA},
                                                      {Value: nfBi("Hearings officers", "Agents d'audience"), Applied: locSessionsCommunityFilters.HO},
                                                      {Value: nfBi("Criminal investigators", "Enquêteurs criminels"), Applied: locSessionsCommunityFilters.CI},
                                                      {Value: nfBi("Intelligence analysts", "Analystes du renseignement"), Applied: locSessionsCommunityFilters.IA},
                                                      {Value: nfBi("Intelligence officers", "Agents du renseignement"), Applied: locSessionsCommunityFilters.IO},
                                                      {Value: nfBi("Enforcement case officers", "Agents de cas d'exécution"), Applied: locSessionsCommunityFilters.ECO},
                                                      {Value: nfBi("Inland enforcement officers", "Agents d'exécution intérieure"), Applied: locSessionsCommunityFilters.IEO},
                                                      {Value: "SOTC", Applied: locSessionsCommunityFilters.SOTC},
                                                      {Value: nfBi("Superintendents", "Surintendants"), Applied: locSessionsCommunityFilters.SUP},
                                                      {Value: nfBi("Management", "Gestion"), Applied: locSessionsCommunityFilters.MGMT},
                                                      {Value: nfBi("Chiefs", "Chefs"), Applied: locSessionsCommunityFilters.CHIEF}
                                                  ),
                                                  Coalesce(Applied, false)
                                              ),
                                              Value,
                                              ", "
                                          ),
                                          nfBi("All", "Toutes")
                                      )
                                    Tooltip: =nfBi("Select target communities", "Sélectionner les communautés cibles")
                              - sessions_cmb_Region:
                                  Control: Classic/ComboBox@2.4.0
                                  Properties:
                                    AccessibleLabel: =nfBi("Filter by region", "Filtrer par région")
                                    BorderColor: =myTheme.Border
                                    BorderThickness: =1
                                    ChevronBackground: =myTheme.Secondary
                                    ChevronDisabledBackground: =myTheme.InputDisabled
                                    ChevronDisabledFill: =myTheme.DisabledText
                                    ChevronFill: =myTheme.SecondaryText
                                    ChevronHoverBackground: =myTheme.SecondaryHover
                                    ChevronHoverFill: =myTheme.SecondaryText
                                    Color: =myTheme.InputText
                                    DefaultSelectedItems: =Blank()
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.InputDisabled
                                    DisplayFields: =["hrbds_code"]
                                    Fill: =myTheme.Input
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Semibold
                                    Height: =48
                                    HoverBorderColor: =myTheme.BorderStrong
                                    HoverColor: =myTheme.InputText
                                    HoverFill: =myTheme.InputHover
                                    InputTextPlaceholder: =nfBi("All regions", "Toutes les régions")
                                    IsSearchable: =false
                                    Items: |-
                                      =Filter(
                                          'CSC Regional Campuses',
                                          'CSC Regional Campuses ID' in nfClientRegions ||
                                          'CSC Regional Campuses ID' in nfUserRegions[@RegionID]
                                      )
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    NoSelectionText: =nfBi("All regions", "Toutes les régions")
                                    OnChange: =Reset(sessions_cmb_District)
                                    PaddingBottom: =6
                                    PaddingLeft: =10
                                    PaddingRight: =10
                                    PaddingTop: =6
                                    PressedBorderColor: =myTheme.Focus
                                    PressedColor: =myTheme.InputText
                                    PressedFill: =myTheme.InputPressed
                                    SearchFields: =["hrbds_code"]
                                    SelectMultiple: =false
                                    SelectionColor: =myTheme.SelectionText
                                    SelectionFill: =myTheme.Selection
                                    SelectionTagColor: =myTheme.SelectionText
                                    SelectionTagFill: =myTheme.Selection
                                    Size: =11
                                    Tooltip: =nfBi("Select a regional campus", "Sélectionner un campus régional")
                              - sessions_cmb_District:
                                  Control: Classic/ComboBox@2.4.0
                                  Properties:
                                    AccessibleLabel: =nfBi("Filter by district or division", "Filtrer par district ou division")
                                    BorderColor: =myTheme.Border
                                    BorderThickness: =1
                                    ChevronBackground: =myTheme.Secondary
                                    ChevronDisabledBackground: =myTheme.InputDisabled
                                    ChevronDisabledFill: =myTheme.DisabledText
                                    ChevronFill: =myTheme.SecondaryText
                                    ChevronHoverBackground: =myTheme.SecondaryHover
                                    ChevronHoverFill: =myTheme.SecondaryText
                                    Color: =myTheme.InputText
                                    DefaultSelectedItems: =Blank()
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.InputDisabled
                                    DisplayFields: =["DisplayValue"]
                                    Fill: =myTheme.Input
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Semibold
                                    Height: =48
                                    HoverBorderColor: =myTheme.BorderStrong
                                    HoverColor: =myTheme.InputText
                                    HoverFill: =myTheme.InputHover
                                    InputTextPlaceholder: =nfBi("All districts or divisions", "Tous les districts ou divisions")
                                    Items: |-
                                      =SortByColumns(
                                          AddColumns(
                                              Filter(
                                                  'CSC Districts Divisions',
                                                  'CSC Districts Divisions ID' in nfUserDistricts[@DistrictID] ||
                                                  'Regional Campus'.'CSC Regional Campuses ID' in nfUserRegions[@RegionID],
                                                  IsBlank(sessions_cmb_Region.Selected.'CSC Regional Campuses ID') ||
                                                  'Regional Campus'.'CSC Regional Campuses ID' =
                                                      sessions_cmb_Region.Selected.'CSC Regional Campuses ID'
                                              ),
                                              DisplayValue,
                                              nfBi(EN, FR)
                                          ),
                                          "hrbds_nameen",
                                          SortOrder.Ascending
                                      )
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    NoSelectionText: =nfBi("All districts or divisions", "Tous les districts ou divisions")
                                    PaddingBottom: =6
                                    PaddingLeft: =10
                                    PaddingRight: =10
                                    PaddingTop: =6
                                    PressedBorderColor: =myTheme.Focus
                                    PressedColor: =myTheme.InputText
                                    PressedFill: =myTheme.InputPressed
                                    SearchFields: =["DisplayValue"]
                                    SelectMultiple: =true
                                    SelectionColor: =myTheme.SelectionText
                                    SelectionFill: =myTheme.Selection
                                    SelectionTagColor: =myTheme.SelectionText
                                    SelectionTagFill: =myTheme.Selection
                                    Size: =11
                                    Tooltip: =nfBi("Select districts or divisions", "Sélectionner des districts ou divisions")
                              - sessions_cmb_Language:
                                  Control: Classic/ComboBox@2.4.0
                                  Properties:
                                    AccessibleLabel: =nfBi("Filter by session language", "Filtrer par langue de la séance")
                                    BorderColor: =myTheme.Border
                                    BorderThickness: =1
                                    ChevronBackground: =myTheme.Secondary
                                    ChevronDisabledBackground: =myTheme.InputDisabled
                                    ChevronDisabledFill: =myTheme.DisabledText
                                    ChevronFill: =myTheme.SecondaryText
                                    ChevronHoverBackground: =myTheme.SecondaryHover
                                    ChevronHoverFill: =myTheme.SecondaryText
                                    Color: =myTheme.InputText
                                    DefaultSelectedItems: =Blank()
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.InputDisabled
                                    DisplayFields: =["DisplayValue"]
                                    Fill: =myTheme.Input
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Semibold
                                    Height: =48
                                    HoverBorderColor: =myTheme.BorderStrong
                                    HoverColor: =myTheme.InputText
                                    HoverFill: =myTheme.InputHover
                                    InputTextPlaceholder: =nfBi("English or French", "Anglais ou français")
                                    IsSearchable: =false
                                    Items: |-
                                      =SortByColumns(
                                          Filter(
                                              nfLookups,
                                              hrbds_lookupcategory = 'CSC Lookup Categories'.Language &&
                                              hrbds_mappingcode in [
                                                  "LANGUAGE-EN",
                                                  "LANGUAGE-FR"
                                              ]
                                          ),
                                          "hrbds_mappingcode",
                                          [
                                              "LANGUAGE-EN",
                                              "LANGUAGE-FR"
                                          ]
                                      )
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    NoSelectionText: =nfBi("English or French", "Anglais ou français")
                                    PaddingBottom: =6
                                    PaddingLeft: =10
                                    PaddingRight: =10
                                    PaddingTop: =6
                                    PressedBorderColor: =myTheme.Focus
                                    PressedColor: =myTheme.InputText
                                    PressedFill: =myTheme.InputPressed
                                    SearchFields: =["DisplayValue"]
                                    SelectMultiple: =true
                                    SelectionColor: =myTheme.SelectionText
                                    SelectionFill: =myTheme.Selection
                                    SelectionTagColor: =myTheme.SelectionText
                                    SelectionTagFill: =myTheme.Selection
                                    Size: =11
                                    Tooltip: =nfBi("Select session languages", "Sélectionner les langues des séances")
                              - sessions_cmb_Category:
                                  Control: Classic/ComboBox@2.4.0
                                  Properties:
                                    AccessibleLabel: =nfBi("Filter by course category", "Filtrer par catégorie de cours")
                                    BorderColor: =myTheme.Border
                                    BorderThickness: =1
                                    ChevronBackground: =myTheme.Secondary
                                    ChevronDisabledBackground: =myTheme.InputDisabled
                                    ChevronDisabledFill: =myTheme.DisabledText
                                    ChevronFill: =myTheme.SecondaryText
                                    ChevronHoverBackground: =myTheme.SecondaryHover
                                    ChevronHoverFill: =myTheme.SecondaryText
                                    Color: =myTheme.InputText
                                    DefaultSelectedItems: =Filter(tbl.Data.Categories, hrbds_code = "ALL")
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.InputDisabled
                                    DisplayFields: =["DisplayValue"]
                                    Fill: =myTheme.Input
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Semibold
                                    Height: =48
                                    HoverBorderColor: =myTheme.BorderStrong
                                    HoverColor: =myTheme.InputText
                                    HoverFill: =myTheme.InputHover
                                    InputTextPlaceholder: =nfBi("All categories", "Toutes les catégories")
                                    IsSearchable: =false
                                    Items: =tbl.Data.Categories
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    NoSelectionText: =nfBi("All categories", "Toutes les catégories")
                                    PaddingBottom: =6
                                    PaddingLeft: =10
                                    PaddingRight: =10
                                    PaddingTop: =6
                                    PressedBorderColor: =myTheme.Focus
                                    PressedColor: =myTheme.InputText
                                    PressedFill: =myTheme.InputPressed
                                    SearchFields: =["DisplayValue"]
                                    SelectMultiple: =false
                                    SelectionColor: =myTheme.SelectionText
                                    SelectionFill: =myTheme.Selection
                                    SelectionTagColor: =myTheme.SelectionText
                                    SelectionTagFill: =myTheme.Selection
                                    Size: =11
                                    Tooltip: =nfBi("Select a course category", "Sélectionner une catégorie de cours")
                              - sessions_btn_ClearFilters:
                                  Control: Classic/Button@2.2.0
                                  Properties:
                                    AlignInContainer: =AlignInContainer.Stretch
                                    BorderColor: =myTheme.Clear
                                    BorderStyle: =BorderStyle.Solid
                                    BorderThickness: =1
                                    Color: =myTheme.ClearText
                                    DisabledBorderColor: =myTheme.Border
                                    DisabledColor: =myTheme.DisabledText
                                    DisabledFill: =myTheme.Disabled
                                    Fill: =myTheme.Clear
                                    FocusedBorderColor: =myTheme.Focus
                                    FocusedBorderThickness: =2
                                    Font: =Font.Lato
                                    FontWeight: =FontWeight.Bold
                                    Height: =42
                                    HoverBorderColor: =myTheme.ClearHover
                                    HoverColor: =myTheme.ClearHoverText
                                    HoverFill: =myTheme.ClearHover
                                    LayoutMinHeight: =42
                                    LayoutMinWidth: =120
                                    OnSelect: |-
                                      =Reset(sessions_txt_Search);
                                      Reset(sessions_cmb_Region);
                                      Reset(sessions_cmb_District);
                                      Reset(sessions_cmb_Language);
                                      Reset(sessions_cmb_Category);
                                      UpdateContext(
                                          {
                                              locSessionsCommunityFilters: Blank(),
                                              locSessionsSelectedSession: Blank()
                                          }
                                      )
                                    PressedBorderColor: =myTheme.ClearPressed
                                    PressedColor: =myTheme.ClearPressedText
                                    PressedFill: =myTheme.ClearPressed
                                    RadiusBottomLeft: =8
                                    RadiusBottomRight: =8
                                    RadiusTopLeft: =8
                                    RadiusTopRight: =8
                                    Size: =11
                                    Text: =nfBi("Clear filters", "Effacer les filtres")
                                    Tooltip: =nfBi("Clear all session filters", "Effacer tous les filtres de séance")
                        - sessions_con_Results:
                            Control: GroupContainer@1.5.0
                            Variant: AutoLayout
                            Properties:
                              DropShadow: =DropShadow.None
                              Fill: =myTheme.Transparent
                              FillPortions: =1
                              LayoutDirection: =LayoutDirection.Vertical
                              LayoutGap: =8
                              LayoutMinHeight: =300
                              LayoutMinWidth: =360
                            Children:
                              - sessions_con_ResultsHeader:
                                  Control: GroupContainer@1.5.0
                                  Variant: AutoLayout
                                  Properties:
                                    DropShadow: =DropShadow.None
                                    Fill: =myTheme.Surface
                                    FillPortions: =0
                                    Height: =48
                                    LayoutAlignItems: =LayoutAlignItems.Center
                                    LayoutDirection: =LayoutDirection.Horizontal
                                    LayoutGap: =8
                                    LayoutMinHeight: =48
                                    LayoutMinWidth: =0
                                    PaddingLeft: =14
                                    PaddingRight: =14
                                    RadiusBottomLeft: =10
                                    RadiusBottomRight: =10
                                    RadiusTopLeft: =10
                                    RadiusTopRight: =10
                                  Children:
                                    - sessions_lbl_ResultsCount:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.Text
                                          Fill: =myTheme.Transparent
                                          FillPortions: =1
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =34
                                          LayoutMinHeight: =34
                                          LayoutMinWidth: =120
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =12
                                          Text: |-
                                            =With(
                                                {_count: sessions_gal_Results.AllItemsCount},
                                                nfBi(
                                                    _count & " session(s) found",
                                                    _count & " séance(s) trouvée(s)"
                                                )
                                            )
                                    - sessions_lbl_SortStatus:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Right
                                          Color: =myTheme.TextMuted
                                          Fill: =myTheme.Transparent
                                          Font: =Font.Lato
                                          Height: =30
                                          LayoutMinHeight: =30
                                          LayoutMinWidth: =130
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =9
                                          Text: |-
                                            =If(
                                                locSessionsSortAscending,
                                                nfBi("Earliest first", "Plus anciennes d'abord"),
                                                nfBi("Latest first", "Plus récentes d'abord")
                                            )
                              - sessions_gal_Results:
                                  Control: Gallery@2.15.0
                                  Variant: Vertical
                                  Properties:
                                    AccessibleLabel: =nfBi("Filtered training sessions", "Séances de formation filtrées")
                                    BorderColor: =myTheme.Transparent
                                    Fill: =myTheme.Transparent
                                    FillPortions: =1
                                    Items: |-
                                      =SortByColumns(
                                          Filter(
                                              AddColumns(
                                                  Filter(
                                                      'CSC Training Sessions' As _session,
                                                      _session.Status = 0,
                                                      _session.'Session Status' =
                                                          'CSC Training Session Statuses'.Scheduled,
                                                      _session.'Registration Period Start Date' <= Today(),
                                                      _session.'End Date' >= Today(),
                                                      IsBlank(Trim(sessions_txt_Search.Text)) ||
                                                      Trim(sessions_txt_Search.Text) in _session.'Session Info' ||
                                                      Trim(sessions_txt_Search.Text) in _session.'Unlisted Course Name' ||
                                                      Trim(sessions_txt_Search.Text) in _session.Course.'Title EN' ||
                                                      Trim(sessions_txt_Search.Text) in _session.Course.'Title FR',
                                                      IsEmpty(sessions_cmb_Language.SelectedItems) ||
                                                      _session.'Session Language'.'CSC Lookups ID' in
                                                          sessions_cmb_Language.SelectedItems.hrbds_csclookupsid ||
                                                      _session.'Session Language'.'Mapping Code' = "LANGUAGE-BIL",
                                                      Coalesce(sessions_cmb_Category.Selected.hrbds_code, "ALL") = "ALL" ||
                                                      _session.Course.'Course Category'.'CSC Lookups ID' =
                                                          sessions_cmb_Category.Selected.hrbds_csclookupsid,
                                                      IsBlank(locSessionsCommunityFilters) ||
                                                      (_session.Course.'Available to BSOs' && locSessionsCommunityFilters.BSO) ||
                                                      (_session.Course.'Available to Chiefs' && locSessionsCommunityFilters.CHIEF) ||
                                                      (_session.Course.'Available to Criminal Investigators' && locSessionsCommunityFilters.CI) ||
                                                      (_session.Course.'Available to Enforcement Case Officers' && locSessionsCommunityFilters.ECO) ||
                                                      (_session.Course.'Available to Hearings Advisors' && locSessionsCommunityFilters.HA) ||
                                                      (_session.Course.'Available to Hearings Officers' && locSessionsCommunityFilters.HO) ||
                                                      (_session.Course.'Available to Inland Enforcement Officers' && locSessionsCommunityFilters.IEO) ||
                                                      (_session.Course.'Available to Intelligence Analysts' && locSessionsCommunityFilters.IA) ||
                                                      (_session.Course.'Available to Intelligence Officers' && locSessionsCommunityFilters.IO) ||
                                                      (_session.Course.'Available to SOTCs' && locSessionsCommunityFilters.SOTC) ||
                                                      (_session.Course.'Available to Superintendents' && locSessionsCommunityFilters.SUP) ||
                                                      (_session.Course.'Available to Supervisors Managers' && locSessionsCommunityFilters.MGMT)
                                                  ) As _source,
                                                  Registrations,
                                                      Filter(
                                                          gblSessionRegistrations As _registration,
                                                          _registration.SessionID =
                                                              _source.'CSC Training Sessions ID',
                                                          IsEmpty(sessions_cmb_District.SelectedItems) ||
                                                          _registration.AssignedDistrict in
                                                              sessions_cmb_District.SelectedItems.'CSC Districts Divisions ID' ||
                                                          (
                                                              _registration.AssignedRegion =
                                                                  sessions_cmb_Region.Selected.'CSC Regional Campuses ID' &&
                                                              IsBlank(_registration.AssignedDistrict)
                                                          ),
                                                          IsBlank(sessions_cmb_Region.Selected.'CSC Regional Campuses ID') ||
                                                          _registration.AssignedRegion =
                                                              sessions_cmb_Region.Selected.'CSC Regional Campuses ID'
                                                      ),
                                                  SeatsTotal,
                                                      Sum(
                                                          Filter(
                                                              gblSessionRegistrations,
                                                              SessionID = _source.'CSC Training Sessions ID'
                                                          ),
                                                          CountSeats
                                                      ),
                                                  SeatsRegistered,
                                                      Sum(
                                                          Filter(
                                                              gblSessionRegistrations,
                                                              SessionID = _source.'CSC Training Sessions ID'
                                                          ),
                                                          CountRegistered
                                                      )
                                              ),
                                              CountRows(Registrations) > 0
                                          ),
                                          "hrbds_startdate",
                                          If(
                                              locSessionsSortAscending,
                                              SortOrder.Ascending,
                                              SortOrder.Descending
                                          )
                                      )
                                    LayoutMinHeight: =250
                                    LayoutMinWidth: =0
                                    TemplateFill: =myTheme.Transparent
                                    TemplatePadding: =6
                                    TemplateSize: =192
                                    Visible: =Self.AllItemsCount > 0
                                    WrapCount: =If(App.Width < 1250, 1, 2)
                                  Children:
                                    - sessions_btn_Card:
                                        Control: Classic/Button@2.2.0
                                        Properties:
                                          BorderColor: |-
                                            =If(
                                                ThisItem.'CSC Training Sessions ID' =
                                                    locSessionsSelectedSession.'CSC Training Sessions ID',
                                                myTheme.Selection,
                                                myTheme.Border
                                            )
                                          BorderStyle: =BorderStyle.Solid
                                          BorderThickness: |-
                                            =If(
                                                ThisItem.'CSC Training Sessions ID' =
                                                    locSessionsSelectedSession.'CSC Training Sessions ID',
                                                2,
                                                1
                                            )
                                          Color: =myTheme.Transparent
                                          Fill: |-
                                            =If(
                                                ThisItem.'CSC Training Sessions ID' =
                                                    locSessionsSelectedSession.'CSC Training Sessions ID',
                                                myTheme.SelectionBackground,
                                                myTheme.Card
                                            )
                                          FocusedBorderColor: =myTheme.Focus
                                          FocusedBorderThickness: =2
                                          Height: =Parent.TemplateHeight - 10
                                          HoverBorderColor: =myTheme.SelectionHover
                                          HoverFill: =myTheme.SurfaceHover
                                          OnSelect: |-
                                            =UpdateContext({locSessionsSelectedSession: ThisItem})
                                          PressedBorderColor: =myTheme.SelectionPressed
                                          PressedFill: =myTheme.SelectionBackground
                                          RadiusBottomLeft: =12
                                          RadiusBottomRight: =12
                                          RadiusTopLeft: =12
                                          RadiusTopRight: =12
                                          Text: =""
                                          Tooltip: |-
                                            =nfBi(
                                                "Select " &
                                                Coalesce(
                                                    ThisItem.'Unlisted Course Name',
                                                    ThisItem.Course.'Title EN'
                                                ),
                                                "Sélectionner " &
                                                Coalesce(
                                                    ThisItem.'Unlisted Course Name',
                                                    ThisItem.Course.'Title FR'
                                                )
                                            )
                                          Width: =Parent.TemplateWidth - 12
                                          X: =6
                                          Y: =4
                                    - sessions_lbl_RegionBadge:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Center
                                          Color: =myTheme.InformationBackgroundText
                                          Fill: =myTheme.InformationBackground
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =24
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =6
                                          PaddingRight: =6
                                          PaddingTop: =0
                                          RadiusBottomLeft: =6
                                          RadiusBottomRight: =6
                                          RadiusTopLeft: =6
                                          RadiusTopRight: =6
                                          Size: =9
                                          Text: =ThisItem.'Coordinating Regional Campus'.Code
                                          Width: =72
                                          X: =16
                                          Y: =14
                                    - sessions_lbl_CodeBadge:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Center
                                          Color: =myTheme.SelectionText
                                          Fill: =myTheme.Selection
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =24
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =6
                                          PaddingRight: =6
                                          PaddingTop: =0
                                          RadiusBottomLeft: =6
                                          RadiusBottomRight: =6
                                          RadiusTopLeft: =6
                                          RadiusTopRight: =6
                                          Size: =9
                                          Text: =ThisItem.Course.Code
                                          Width: =90
                                          X: =96
                                          Y: =14
                                    - sessions_lbl_LanguageBadge:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Center
                                          Color: =myTheme.AccentText
                                          Fill: =myTheme.Accent
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =24
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =4
                                          PaddingRight: =4
                                          PaddingTop: =0
                                          RadiusBottomLeft: =6
                                          RadiusBottomRight: =6
                                          RadiusTopLeft: =6
                                          RadiusTopRight: =6
                                          Size: =9
                                          Text: |-
                                            =Switch(
                                                ThisItem.'Session Language'.'Mapping Code',
                                                "LANGUAGE-EN", nfBi("EN", "AN"),
                                                "LANGUAGE-FR", "FR",
                                                "LANGUAGE-BIL", "BIL",
                                                "–"
                                            )
                                          Width: =44
                                          X: =Parent.TemplateWidth - 64
                                          Y: =14
                                    - sessions_lbl_CourseTitle:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.Text
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =42
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =8
                                          PaddingTop: =0
                                          Size: =12
                                          Text: |-
                                            =Coalesce(
                                                ThisItem.'Unlisted Course Name',
                                                nfBi(
                                                    ThisItem.Course.'Title EN',
                                                    ThisItem.Course.'Title FR'
                                                )
                                            )
                                          VerticalAlign: =VerticalAlign.Middle
                                          Width: =Parent.TemplateWidth - 32
                                          X: =16
                                          Y: =45
                                    - sessions_ico_Date:
                                        Control: Classic/Icon@2.5.0
                                        Properties:
                                          AccessibleLabel: =nfBi("Session date", "Date de la séance")
                                          Color: =myTheme.IconsMuted
                                          Height: =22
                                          HoverColor: =myTheme.IconsMuted
                                          Icon: =Icon.CalendarBlank
                                          OnSelect: =Select(sessions_btn_Card)
                                          PressedColor: =myTheme.IconsMuted
                                          TabIndex: =-1
                                          Width: =22
                                          X: =16
                                          Y: =94
                                    - sessions_lbl_Date:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.TextMuted
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Semibold
                                          Height: =26
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =10
                                          Text: |-
                                            =Text(
                                                ThisItem.'Start Date',
                                                If(
                                                    ShowFrench,
                                                    "[$-fr-CA]d mmm yyyy",
                                                    "[$-en-CA]mmm d, yyyy"
                                                )
                                            ) &
                                            If(
                                                !IsBlank(ThisItem.'End Date') &&
                                                ThisItem.'Start Date' <> ThisItem.'End Date',
                                                nfBi(" to ", " au ") &
                                                Text(
                                                    ThisItem.'End Date',
                                                    If(
                                                        ShowFrench,
                                                        "[$-fr-CA]d mmm yyyy",
                                                        "[$-en-CA]mmm d, yyyy"
                                                    )
                                                ),
                                                ""
                                            )
                                          Width: =Parent.TemplateWidth - 65
                                          X: =46
                                          Y: =92
                                    - sessions_ico_Location:
                                        Control: Classic/Icon@2.5.0
                                        Properties:
                                          AccessibleLabel: =nfBi("Session location", "Emplacement de la séance")
                                          Color: =myTheme.IconsMuted
                                          Height: =22
                                          HoverColor: =myTheme.IconsMuted
                                          Icon: =Icon.Waypoint
                                          OnSelect: =Select(sessions_btn_Card)
                                          PressedColor: =myTheme.IconsMuted
                                          TabIndex: =-1
                                          Width: =22
                                          X: =16
                                          Y: =123
                                    - sessions_lbl_Location:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: =myTheme.TextMuted
                                          Font: =Font.Lato
                                          Height: =26
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =0
                                          PaddingRight: =0
                                          PaddingTop: =0
                                          Size: =10
                                          Text: |-
                                            =If(
                                                ThisItem.'Virtual Delivery' =
                                                    'Virtual Delivery (CSC Training Sessions)'.Yes,
                                                nfBi("Virtual", "Virtuelle"),
                                                Coalesce(
                                                    ThisItem.'Session City' &
                                                    If(
                                                        !IsBlank(ThisItem.'Session Province'.'CSC Lookups ID'),
                                                        ", " &
                                                        nfBi(
                                                            ThisItem.'Session Province'.EN,
                                                            ThisItem.'Session Province'.FR
                                                        ),
                                                        ""
                                                    ),
                                                    "–"
                                                )
                                            )
                                          Width: =Parent.TemplateWidth - 65
                                          X: =46
                                          Y: =121
                                    - sessions_lbl_Capacity:
                                        Control: Label@2.5.1
                                        Properties:
                                          Color: |-
                                            =With(
                                                {
                                                    _remaining:
                                                        ThisItem.SeatsTotal -
                                                        ThisItem.SeatsRegistered
                                                },
                                                If(
                                                    ThisItem.'Registration Period End Date' < Today(),
                                                    myTheme.InformationBackgroundText,
                                                    _remaining <= 0,
                                                    myTheme.ErrorBackgroundText,
                                                    _remaining <= 4,
                                                    myTheme.WarningBackgroundText,
                                                    myTheme.SuccessBackgroundText
                                                )
                                            )
                                          Fill: |-
                                            =With(
                                                {
                                                    _remaining:
                                                        ThisItem.SeatsTotal -
                                                        ThisItem.SeatsRegistered
                                                },
                                                If(
                                                    ThisItem.'Registration Period End Date' < Today(),
                                                    myTheme.InformationBackground,
                                                    _remaining <= 0,
                                                    myTheme.ErrorBackground,
                                                    _remaining <= 4,
                                                    myTheme.WarningBackground,
                                                    myTheme.SuccessBackground
                                                )
                                            )
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =28
                                          OnSelect: =Select(sessions_btn_Card)
                                          PaddingBottom: =0
                                          PaddingLeft: =8
                                          PaddingRight: =8
                                          PaddingTop: =0
                                          RadiusBottomLeft: =6
                                          RadiusBottomRight: =6
                                          RadiusTopLeft: =6
                                          RadiusTopRight: =6
                                          Size: =9
                                          Text: |-
                                            =With(
                                                {
                                                    _remaining:
                                                        ThisItem.SeatsTotal -
                                                        ThisItem.SeatsRegistered
                                                },
                                                If(
                                                    ThisItem.'Registration Period End Date' < Today(),
                                                    nfBi("Registration closed", "Inscription fermée"),
                                                    _remaining <= 0,
                                                    nfBi("Full", "Complet"),
                                                    _remaining <= 4,
                                                    nfBi(
                                                        _remaining & " remaining",
                                                        _remaining & " restante(s)"
                                                    ),
                                                    nfBi(
                                                        ThisItem.SeatsRegistered & " / " &
                                                        ThisItem.SeatsTotal & " registered",
                                                        ThisItem.SeatsRegistered & " / " &
                                                        ThisItem.SeatsTotal & " inscrit(s)"
                                                    )
                                                )
                                            )
                                          Width: =190
                                          X: =16
                                          Y: =153
                                    - sessions_btn_Details:
                                        Control: Classic/Button@2.2.0
                                        Properties:
                                          BorderColor: =myTheme.Secondary
                                          BorderStyle: =BorderStyle.Solid
                                          BorderThickness: =1
                                          Color: =myTheme.Secondary
                                          Fill: =myTheme.Transparent
                                          FocusedBorderColor: =myTheme.Focus
                                          FocusedBorderThickness: =2
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =30
                                          HoverBorderColor: =myTheme.SecondaryHover
                                          HoverColor: =myTheme.SecondaryText
                                          HoverFill: =myTheme.SecondaryHover
                                          OnSelect: |-
                                            =Navigate(
                                                'Session Details | Détails de la séance',
                                                ScreenTransition.Cover,
                                                {
                                                    locTrainingSessionID:
                                                        ThisItem.'CSC Training Sessions ID'
                                                }
                                            )
                                          PressedBorderColor: =myTheme.SecondaryPressed
                                          PressedColor: =myTheme.SecondaryText
                                          PressedFill: =myTheme.SecondaryPressed
                                          RadiusBottomLeft: =6
                                          RadiusBottomRight: =6
                                          RadiusTopLeft: =6
                                          RadiusTopRight: =6
                                          Size: =9
                                          Text: =nfBi("View details", "Voir les détails")
                                          Tooltip: =nfBi("Open session details", "Ouvrir les détails de la séance")
                                          Width: =110
                                          X: =Parent.TemplateWidth - 130
                                          Y: =152
                              - sessions_con_EmptyState:
                                  Control: GroupContainer@1.5.0
                                  Variant: AutoLayout
                                  Properties:
                                    AlignInContainer: =AlignInContainer.Stretch
                                    DropShadow: =DropShadow.None
                                    Fill: =myTheme.InformationBackground
                                    FillPortions: =1
                                    LayoutAlignItems: =LayoutAlignItems.Center
                                    LayoutDirection: =LayoutDirection.Vertical
                                    LayoutGap: =8
                                    LayoutJustifyContent: =LayoutJustifyContent.Center
                                    LayoutMinHeight: =180
                                    LayoutMinWidth: =0
                                    RadiusBottomLeft: =12
                                    RadiusBottomRight: =12
                                    RadiusTopLeft: =12
                                    RadiusTopRight: =12
                                    Visible: =sessions_gal_Results.AllItemsCount = 0
                                  Children:
                                    - sessions_ico_Empty:
                                        Control: Classic/Icon@2.5.0
                                        Properties:
                                          AccessibleLabel: =nfBi("No sessions found", "Aucune séance trouvée")
                                          Color: =myTheme.InformationBackgroundText
                                          Height: =48
                                          HoverColor: =myTheme.InformationBackgroundText
                                          Icon: =Icon.Search
                                          LayoutMinHeight: =48
                                          LayoutMinWidth: =48
                                          PressedColor: =myTheme.InformationBackgroundText
                                          TabIndex: =0
                                          Tooltip: =nfBi("No sessions found", "Aucune séance trouvée")
                                          Width: =48
                                    - sessions_lbl_EmptyTitle:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Center
                                          Color: =myTheme.InformationBackgroundText
                                          Fill: =myTheme.Transparent
                                          Font: =Font.Lato
                                          FontWeight: =FontWeight.Bold
                                          Height: =34
                                          LayoutMinHeight: =34
                                          LayoutMinWidth: =240
                                          Size: =14
                                          Text: =nfBi("No matching sessions", "Aucune séance correspondante")
                                          Width: =Min(520, Parent.Width - 40)
                                    - sessions_lbl_EmptyDescription:
                                        Control: Label@2.5.1
                                        Properties:
                                          Align: =Align.Center
                                          Color: =myTheme.InformationBackgroundText
                                          Fill: =myTheme.Transparent
                                          Font: =Font.Lato
                                          Height: =52
                                          LayoutMinHeight: =52
                                          LayoutMinWidth: =240
                                          Size: =10
                                          Text: |-
                                            =nfBi(
                                                "Adjust your filters or contact your regional training coordinator for assistance.",
                                                "Modifiez vos filtres ou communiquez avec votre coordonnateur régional de la formation pour obtenir de l'aide."
                                            )
                                          Width: =Min(600, Parent.Width - 40)
                  - sessions_con_Footer:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Surface
                        FillPortions: =0
                        Height: =38
                        LayoutAlignItems: =LayoutAlignItems.Center
                        LayoutDirection: =LayoutDirection.Horizontal
                        LayoutGap: =8
                        LayoutMinHeight: =38
                        LayoutMinWidth: =0
                        PaddingLeft: =12
                        PaddingRight: =12
                        RadiusBottomLeft: =10
                        RadiusBottomRight: =10
                        RadiusTopLeft: =10
                        RadiusTopRight: =10
                      Children:
                        - sessions_lbl_Refresh:
                            Control: Label@2.5.1
                            Properties:
                              Color: =myTheme.TextMuted
                              Fill: =myTheme.Transparent
                              FillPortions: =1
                              Font: =Font.Lato
                              Height: =28
                              LayoutMinHeight: =28
                              LayoutMinWidth: =100
                              PaddingBottom: =0
                              PaddingLeft: =0
                              PaddingRight: =0
                              PaddingTop: =0
                              Size: =9
                              Text: |-
                                =nfBi("Results calculated ", "Résultats calculés ") &
                                Text(
                                    locSessionsRefreshedAt,
                                    If(
                                        ShowFrench,
                                        "[$-fr-CA]yyyy-mm-dd HH:mm",
                                        "[$-en-CA]yyyy-mm-dd HH:mm"
                                    )
                                )
                        - sessions_lbl_CurrentUser:
                            Control: Label@2.5.1
                            Properties:
                              Align: =Align.Right
                              Color: =myTheme.TextMuted
                              Fill: =myTheme.Transparent
                              Font: =Font.Lato
                              FontWeight: =FontWeight.Semibold
                              Height: =28
                              LayoutMinHeight: =28
                              LayoutMinWidth: =180
                              PaddingBottom: =0
                              PaddingLeft: =0
                              PaddingRight: =0
                              PaddingTop: =0
                              Size: =9
                              Text: |-
                                =nfBi("Current user: ", "Utilisateur actuel : ") & User().FullName
      - sessions_con_CommunityOverlay:
          Control: GroupContainer@1.5.0
          Variant: ManualLayout
          Properties:
            DropShadow: =DropShadow.None
            Fill: =myTheme.Overlay
            Height: =App.Height
            Visible: =locSessionsShowCommunities
            Width: =App.Width
          Children:
            - sessions_con_CommunityDialog:
                Control: GroupContainer@1.5.0
                Variant: AutoLayout
                Properties:
                  DropShadow: =DropShadow.Regular
                  Fill: =myTheme.Surface
                  Height: =Min(620, Parent.Height - 48)
                  LayoutDirection: =LayoutDirection.Vertical
                  LayoutGap: =10
                  PaddingBottom: =20
                  PaddingLeft: =20
                  PaddingRight: =20
                  PaddingTop: =16
                  RadiusBottomLeft: =14
                  RadiusBottomRight: =14
                  RadiusTopLeft: =14
                  RadiusTopRight: =14
                  Width: =Min(850, Parent.Width - 48)
                  X: =(Parent.Width - Self.Width) / 2
                  Y: =(Parent.Height - Self.Height) / 2
                Children:
                  - sessions_con_CommunityHeader:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Transparent
                        FillPortions: =0
                        Height: =48
                        LayoutAlignItems: =LayoutAlignItems.Center
                        LayoutDirection: =LayoutDirection.Horizontal
                        LayoutGap: =10
                        LayoutMinHeight: =48
                        LayoutMinWidth: =0
                      Children:
                        - sessions_lbl_CommunityTitle:
                            Control: Label@2.5.1
                            Properties:
                              Color: =myTheme.Text
                              Fill: =myTheme.Transparent
                              FillPortions: =1
                              Font: =Font.Lato
                              FontWeight: =FontWeight.Bold
                              Height: =40
                              LayoutMinHeight: =40
                              LayoutMinWidth: =170
                              PaddingBottom: =0
                              PaddingLeft: =0
                              PaddingRight: =0
                              PaddingTop: =0
                              Size: =18
                              Text: =nfBi("Target communities", "Communautés cibles")
                        - sessions_ico_CloseCommunity:
                            Control: Classic/Icon@2.5.0
                            Properties:
                              AccessibleLabel: =nfBi("Close community filters", "Fermer les filtres de communauté")
                              Color: =myTheme.Icons
                              Height: =36
                              HoverColor: =myTheme.SecondaryHover
                              Icon: =Icon.Cancel
                              LayoutMinHeight: =36
                              LayoutMinWidth: =36
                              OnSelect: |-
                                =UpdateContext({locSessionsShowCommunities: false})
                              PressedColor: =myTheme.SecondaryPressed
                              TabIndex: =0
                              Tooltip: =nfBi("Close", "Fermer")
                              Width: =36
                  - sessions_lbl_CommunityHelp:
                      Control: Label@2.5.1
                      Properties:
                        Color: =myTheme.TextMuted
                        Fill: =myTheme.Transparent
                        FillPortions: =0
                        Font: =Font.Lato
                        Height: =36
                        LayoutMinHeight: =36
                        LayoutMinWidth: =0
                        PaddingBottom: =0
                        PaddingLeft: =0
                        PaddingRight: =0
                        PaddingTop: =0
                        Size: =10
                        Text: |-
                          =nfBi(
                              "Select one or more communities. Leave all options cleared to include every community.",
                              "Sélectionnez une ou plusieurs communautés. Laissez toutes les options décochées pour inclure toutes les communautés."
                          )
                  - sessions_con_CommunityOptions:
                      Control: GroupContainer@1.5.0
                      Variant: GridLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.SurfaceSubtle
                        FillPortions: =1
                        LayoutGap: =6
                        LayoutGridColumnMinWidth: =220
                        LayoutGridColumns: =If(App.Width < 800, 1, 3)
                        LayoutGridRowMinHeight: =52
                        LayoutMinHeight: =300
                        LayoutMinWidth: =0
                        PaddingBottom: =12
                        PaddingLeft: =12
                        PaddingRight: =12
                        PaddingTop: =12
                        RadiusBottomLeft: =10
                        RadiusBottomRight: =10
                        RadiusTopLeft: =10
                        RadiusTopRight: =10
                      Children:
                        - sessions_chk_BSO:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.BSO, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Border Services Officers", "Agents des services frontaliers")
                              Tooltip: =nfBi("Border Services Officers", "Agents des services frontaliers")
                        - sessions_chk_HA:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.HA, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Hearings Advisors", "Conseillers aux audiences")
                        - sessions_chk_HO:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.HO, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Hearings Officers", "Agents d'audience")
                        - sessions_chk_CI:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.CI, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Criminal Investigators", "Enquêteurs criminels")
                        - sessions_chk_IA:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.IA, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Intelligence Analysts", "Analystes du renseignement")
                        - sessions_chk_IO:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.IO, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Intelligence Officers", "Agents du renseignement")
                        - sessions_chk_ECO:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.ECO, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Enforcement Case Officers", "Agents de cas d'exécution")
                        - sessions_chk_IEO:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.IEO, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Inland Enforcement Officers", "Agents d'exécution intérieure")
                        - sessions_chk_SOTC:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.SOTC, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Senior Trade Compliance Officers", "Agents principaux de conformité commerciale")
                        - sessions_chk_SUP:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.SUP, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Superintendents", "Surintendants")
                        - sessions_chk_MGMT:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.MGMT, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Supervisors and Managers", "Superviseurs et gestionnaires")
                        - sessions_chk_CHIEF:
                            Control: Classic/CheckBox@2.1.0
                            Properties:
                              CheckboxBackgroundFill: =myTheme.Input
                              CheckboxBorderColor: =myTheme.BorderStrong
                              CheckboxSize: =28
                              CheckmarkFill: =myTheme.Selection
                              Color: =myTheme.Text
                              Default: =Coalesce(locSessionsCommunityFilters.CHIEF, false)
                              Fill: =myTheme.Transparent
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              Height: =48
                              Size: =10
                              Text: =nfBi("Chiefs", "Chefs")
                  - sessions_con_CommunityActions:
                      Control: GroupContainer@1.5.0
                      Variant: AutoLayout
                      Properties:
                        DropShadow: =DropShadow.None
                        Fill: =myTheme.Transparent
                        FillPortions: =0
                        Height: =48
                        LayoutAlignItems: =LayoutAlignItems.Center
                        LayoutDirection: =LayoutDirection.Horizontal
                        LayoutGap: =10
                        LayoutJustifyContent: =LayoutJustifyContent.End
                        LayoutMinHeight: =48
                        LayoutMinWidth: =0
                      Children:
                        - sessions_btn_ClearCommunities:
                            Control: Classic/Button@2.2.0
                            Properties:
                              BorderColor: =myTheme.Clear
                              BorderStyle: =BorderStyle.Solid
                              BorderThickness: =1
                              Color: =myTheme.ClearText
                              Fill: =myTheme.Clear
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              FontWeight: =FontWeight.Bold
                              Height: =40
                              HoverBorderColor: =myTheme.ClearHover
                              HoverColor: =myTheme.ClearHoverText
                              HoverFill: =myTheme.ClearHover
                              LayoutMinHeight: =40
                              LayoutMinWidth: =130
                              OnSelect: |-
                                =UpdateContext(
                                    {
                                        locSessionsShowCommunities: false,
                                        locSessionsCommunityFilters: Blank()
                                    }
                                )
                              PressedBorderColor: =myTheme.ClearPressed
                              PressedColor: =myTheme.ClearPressedText
                              PressedFill: =myTheme.ClearPressed
                              RadiusBottomLeft: =8
                              RadiusBottomRight: =8
                              RadiusTopLeft: =8
                              RadiusTopRight: =8
                              Size: =10
                              Text: =nfBi("Clear all", "Tout effacer")
                              Tooltip: =nfBi("Clear community filters", "Effacer les filtres de communauté")
                              Width: =130
                        - sessions_btn_ApplyCommunities:
                            Control: Classic/Button@2.2.0
                            Properties:
                              BorderColor: =myTheme.Primary
                              BorderStyle: =BorderStyle.Solid
                              BorderThickness: =1
                              Color: =myTheme.PrimaryText
                              Fill: =myTheme.Primary
                              FocusedBorderColor: =myTheme.Focus
                              FocusedBorderThickness: =2
                              Font: =Font.Lato
                              FontWeight: =FontWeight.Bold
                              Height: =40
                              HoverBorderColor: =myTheme.PrimaryHover
                              HoverColor: =myTheme.PrimaryText
                              HoverFill: =myTheme.PrimaryHover
                              LayoutMinHeight: =40
                              LayoutMinWidth: =130
                              OnSelect: |-
                                =UpdateContext(
                                    {
                                        locSessionsShowCommunities: false,
                                        locSessionsCommunityFilters:
                                            If(
                                                sessions_chk_BSO.Value ||
                                                sessions_chk_HA.Value ||
                                                sessions_chk_HO.Value ||
                                                sessions_chk_CI.Value ||
                                                sessions_chk_IA.Value ||
                                                sessions_chk_IO.Value ||
                                                sessions_chk_ECO.Value ||
                                                sessions_chk_IEO.Value ||
                                                sessions_chk_SOTC.Value ||
                                                sessions_chk_SUP.Value ||
                                                sessions_chk_MGMT.Value ||
                                                sessions_chk_CHIEF.Value,
                                                {
                                                    BSO: Coalesce(sessions_chk_BSO.Value, false),
                                                    HA: Coalesce(sessions_chk_HA.Value, false),
                                                    HO: Coalesce(sessions_chk_HO.Value, false),
                                                    CI: Coalesce(sessions_chk_CI.Value, false),
                                                    IA: Coalesce(sessions_chk_IA.Value, false),
                                                    IO: Coalesce(sessions_chk_IO.Value, false),
                                                    ECO: Coalesce(sessions_chk_ECO.Value, false),
                                                    IEO: Coalesce(sessions_chk_IEO.Value, false),
                                                    SOTC: Coalesce(sessions_chk_SOTC.Value, false),
                                                    SUP: Coalesce(sessions_chk_SUP.Value, false),
                                                    MGMT: Coalesce(sessions_chk_MGMT.Value, false),
                                                    CHIEF: Coalesce(sessions_chk_CHIEF.Value, false)
                                                },
                                                Blank()
                                            )
                                    }
                                )
                              PressedBorderColor: =myTheme.PrimaryPressed
                              PressedColor: =myTheme.PrimaryText
                              PressedFill: =myTheme.PrimaryPressed
                              RadiusBottomLeft: =8
                              RadiusBottomRight: =8
                              RadiusTopLeft: =8
                              RadiusTopRight: =8
                              Size: =10
                              Text: =nfBi("Apply filters", "Appliquer les filtres")
                              Tooltip: =nfBi("Apply community filters", "Appliquer les filtres de communauté")
                              Width: =140
"""
baseline = validate_text(sample_yaml, "sample.pa.yaml")

pa2108_playground = """Screens:
  PA2108 Demo:
    Children:
      - lbl_rounded:
          Control: Label@2.5.1
          Properties:
            Text: =nfBi("Rounded label trap", "Piège libellé arrondi")
            RadiusTopLeft: =12
            RadiusTopRight: =12
            RadiusBottomLeft: =12
            RadiusBottomRight: =12
            Size: =14
"""

print(f"Sample baseline: {len(baseline)} finding(s)")
print(f"PA2108 playground: {[d.code for d in validate_text(pa2108_playground)]}")


  error: subprocess-exited-with-error
  
  × Building wheel for matplotlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [1100 lines of output]
      
      Edit mplsetup.cfg to change the build options; suppress output with --quiet.
      
      BUILDING MATPLOTLIB
            python: yes [3.14.5 (main, May 10 2026, 10:21:34) [Clang 21.0.0
                        (clang-2100.0.123.102)]]
          platform: yes [darwin]
             tests: no  [skipping due to configuration]
            macosx: yes [installing]
      
      /private/var/folders/jx/mcyjq1zs6f19z7lrxr_tvygm0000gp/T/pip-build-env-b3ak45cg/overlay/lib/python3.14/site-packages/vcs_versioning/_fallback_workdir.py:248: UserWarning: git archive did not support describe output
        return archival_to_version(data, config)
      /private/var/folders/jx/mcyjq1zs6f19z7lrxr_tvygm0000gp/T/pip-build-env-b3ak45cg/overlay/lib/python3.14/site-packages/vcs_versioning/_fallback_workdir.py:248: UserWarning: unproces

In [5]:
import importlib
import powerapps_yaml_validator

importlib.reload(powerapps_yaml_validator)

# Load the training-sessions sample, or paste pa2108_playground to exercise Radius-on-Label repairs.
ui = powerapps_yaml_validator.create_validator_ui(
    sample_yaml,
    source_name="training-sessions.pa.yaml",
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>